# AI-Powered Fake Login Page and phishing Detection using Machine Learning

This Jupyter Notebook serves as the Exploratory Data Analysis (EDA) and interactive training playground for our 3rd-year B.Tech project. We will use the **PhiUSIIL Phishing URL Dataset** to train a Random Forest model that will be exported for browser extension integration.

## Project Goals:
1. **Load and Inspect** the PhiUSIIL Phishing URL Dataset.
2. **Exploratory Data Analysis (EDA)** to understand feature behaviors and check class balance.
3. **Preprocess and Clean** the dataset (handling nulls, duplicates, and encoding categorical variables).
4. **Train and Compare** Machine Learning models (Decision Tree, Random Forest, XGBoost).
5. **Save the Best Model** and Encoders for production deployment.

In [ ]:
# Install dependencies if not already installed
# !pip install -r ../requirements.txt

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Allow importing from src folder
sys.path.append(os.path.abspath(os.path.join('..')))

from src.preprocessing import load_data, inspect_data, preprocess_pipeline, split_and_prepare
from src.evaluate import calculate_metrics, plot_confusion_matrix, plot_roc_curve, plot_feature_importance, plot_correlation_heatmap
from src.utils import setup_logger

logger = setup_logger(log_dir="../logs")
logger.info("Notebook environment configured successfully.")

## 1. Load and Inspect the Dataset

In [ ]:
# Define path to the dataset
dataset_path = os.path.join("..", "dataset", "PhiUSIIL_Phishing_URL_Dataset.csv")

# Fallback if file is in the root directory
if not os.path.exists(dataset_path) and os.path.exists(os.path.join("..", "PhiUSIIL_Phishing_URL_Dataset.csv")):
    dataset_path = os.path.join("..", "PhiUSIIL_Phishing_URL_Dataset.csv")

df = load_data(dataset_path, logger)
df.head()

In [ ]:
# Inspect shape, duplicate records, and target class distributions
inspect_data(df, logger)

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# 2.1 Target Class Balance
plt.figure(figsize=(6, 4))
sns.countplot(x='label', data=df, hue='label', palette='Set2', legend=False)
plt.title('Target Class Distribution (0 = Phishing, 1 = Legitimate)')
plt.xlabel('Label')
plt.ylabel('Count')
plt.show()

In [ ]:
# 2.2 Numerical features distribution comparison
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.boxplot(x='label', y='URLLength', data=df, hue='label', palette='Set2', legend=False)
plt.title('URL Length Distribution')

plt.subplot(1, 2, 2)
sns.boxplot(x='label', y='DomainLength', data=df, hue='label', palette='Set2', legend=False)
plt.title('Domain Length Distribution')
plt.show()

## 3. Preprocessing and Feature Selection

In [ ]:
# Drops non-numerical metadata (URL, Domain, Title) and LabelEncodes TLD
X, y, encoders = preprocess_pipeline(df, logger, target_col='label')
X.head()

In [ ]:
# Plot the correlation matrix for top high-variance features
plot_correlation_heatmap(X, save_dir="../reports")

# Display correlation heatmap directly in notebook
corr = X.select_dtypes(include=[np.number]).corr()
plt.figure(figsize=(14, 12))
sns.heatmap(corr, annot=False, cmap='coolwarm', linewidths=0.2)
plt.title('Correlation Matrix of All Features')
plt.show()

## 4. Train-Test Split (80/20 Stratified)

In [ ]:
X_train, X_test, y_train, y_test = split_and_prepare(X, y, logger, test_size=0.2, random_state=42)

## 5. Train Random Forest Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import time

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

logger.info("Training Random Forest...")
start_time = time.time()
rf_model.fit(X_train, y_train)
logger.info(f"Training finished in {time.time() - start_time:.2f} seconds.")

## 6. Evaluate Random Forest Model

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = rf_model.predict(X_test)
print("Random Forest Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=["Phishing", "Legitimate"]))

# Display Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
            xticklabels=["Phishing (0)", "Legitimate (1)"],
            yticklabels=["Phishing (0)", "Legitimate (1)"])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Random Forest Confusion Matrix')
plt.show()

In [ ]:
# Plot Feature Importances
plot_feature_importance(rf_model, X.columns.tolist(), "Random Forest", save_dir="../reports", top_n=20)

# Display Top 15 importances in the notebook
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1][:15]
features = X.columns.tolist()

plt.figure(figsize=(10, 6))
sns.barplot(x=importances[indices], y=[features[i] for i in indices], palette='viridis', hue=[features[i] for i in indices], legend=False)
plt.title('Top 15 Feature Importances')
plt.xlabel('Score')
plt.ylabel('Feature')
plt.show()

## 7. Conclusions

The Random Forest model displays exceptionally high performance on the PhiUSIIL URL dataset due to features like `URLSimilarityIndex`, `URLLength`, `IsHTTPS`, `HasPasswordField`, and `NoOfiFrame` that directly separate phishing indicators from legitimate sites. 

To proceed with model deployment, run the main orchestrator script in your terminal:
```bash
python src/train.py
```
This will train all model configurations (Decision Tree, Random Forest, XGBoost), generate reports, select the best model, and save it in the `models/` directory.